Link base do github: https://github.com/glaucogoncalves/p-e/tree/main/exames/exame-b-bike-sharing

## O objetivo final:
O desafio é investigar um conjunto de dados sobre o sistema de bicicletas compartilhadas de Seul, na Coreia do Sul, e, a partir dele, desenvolver um modelo capaz de estimar quantas bicicletas serão alugadas em uma determinada hora, com base nas condições do tempo e do calendário.

- **Sugestões do que investigar:**
  - Medidas resumo (média, mediana, desvio padrão, assimetria) para as variáveis numéricas.
  - Tabelas de frequência para as variáveis categóricas, apontando categorias dominantes e pouco representadas.
  - Histogramas e *boxplots* para visualizar o formato das distribuições e localizar possíveis *outliers*.
  - Verificação da qualidade dos dados (valores ausentes, duplicados, valores implausíveis), com proposta de tratamento.
  - Gráficos de dispersão entre `Rented_Bike_Count` e as variáveis contínuas.
  - *Boxplots* ou *violin plots* para comparar a distribuição de `Rented_Bike_Count` entre as categorias de uma variável.
  - Matriz de correlação entre as variáveis numéricas, com interpretação dos resultados.
  - Um fechamento destacando quais variáveis parecem mais promissoras para o modelo.

- **O que entregar:**
  - Notebook com as análises, gráficos e comentários explicando cada passo e o que foi observado.






## Overview dos dados:
Este conjunto de dados reúne a quantidade de bicicletas alugadas a cada hora no sistema público de bicicletas compartilhadas de Seul (Coreia do Sul), junto com as condições meteorológicas registradas no mesmo horário e informações sobre o calendário (estação do ano, feriados e funcionamento do sistema).

O objetivo do estudo é entender como o clima e o calendário se relacionam com a demanda por bicicletas e construir um modelo capaz de prever o número de bicicletas alugadas em uma determinada hora (Rented_Bike_Count).

- Período coberto: 01/12/2017 a 30/11/2018.
- Total de observações: 7.008 registros horários.
- Total de variáveis: 14.
- Variável dependente: 1 (Rented_Bike_Count).
- Variáveis de identificação/tempo: 2 (Date e Hour).
- Variáveis preditoras: 11 (8 numéricas + 3 categóricas), além de Date e Hour.


## Variáveis de Tempo

| Nome da Variável | Descrição | Tipo |
| :--- | :--- | :--- |
| `Date` | Data do registro, no formato `DD/MM/AAAA`. | Data |
| `Hour` | Hora do dia a que o registro se refere (0 a 23). | Numérica (Discreta) |

> **Nota:** Cada linha corresponde a uma hora de um dia. A coluna `Date` não deve ser usada diretamente no modelo, mas pode servir de base para criar novas variáveis.

---

## Variável Dependente (Alvo)

| Nome da Variável | Descrição | Tipo |
| :--- | :--- | :--- |
| `Rented_Bike_Count` | Quantidade de bicicletas alugadas naquela hora. Varia de 0 a 3.556, com média de 704,97 e mediana de 504,5. | Numérica (Discreta) |

---

## Variáveis Preditoras

### Dados Meteorológicos

| Nome da Variável | Descrição | Unidade | Tipo |
| :--- | :--- | :--- | :--- |
| `Temperature` | Temperatura do ar. Varia de -17,8 a 39,4 (média 12,91). | °C | Numérica |
| `Humidity` | Umidade relativa do ar. Varia de 0 a 98 (média 58,27). | % | Numérica |
| `Wind_Speed` | Velocidade do vento. Varia de 0 a 7,4 (média 1,73). | m/s | Numérica |
| `Visibility` | Visibilidade horizontal. Varia de 27 a 2.000 (média 1.434,96). | unidades de 10 m | Numérica |
| `Dew_Point_Temperature` | Temperatura do ponto de orvalho, ou seja, a temperatura em que o vapor d'água do ar começa a condensar. Varia de -30,6 a 26,8 (média 4,12). | °C | Numérica |
| `Solar_Radiation` | Radiação solar incidente. Varia de 0 a 3,52 (média 0,57). | MJ/m² | Numérica |
| `Rainfall` | Volume de chuva na hora. Varia de 0 a 29,5 (média 0,14). | mm | Numérica |
| `Snowfall` | Volume de neve na hora. Varia de 0 a 8,8 (média 0,08). | cm | Numérica |

### Dados de Calendário

| Nome da Variável | Descrição | Tipo |
| :--- | :--- | :--- |
| `Seasons` | Estação do ano. Possui 4 categorias: `Spring` (1.766), `Summer` (1.767), `Autumn` (1.747) e `Winter` (1.728). | Categórica |
| `Holiday` | Indica se o dia é feriado. Possui 2 categorias: `No Holiday` (6.660) e `Holiday` (348). | Categórica (Binária) |
| `Functioning_Day` | Indica se o sistema de bicicletas estava em funcionamento naquele dia. Possui 2 categorias: `Yes` (6.774) e `No` (234). | Categórica (Binária) |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/glaucogoncalves/p-e/refs/heads/main/exames/exame-b-bike-sharing/bike_sharing.csv")

In [ ]:
df.info()

Trabalhar com uma data assim é ruim para ordenar

In [ ]:
df['Date_Parsed'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df.sort_values(by="Date_Parsed")

Coisas para se pensar:
Porque uma pessoa quer se deslocar mais rápido?

Aqui eu acho que dois fatores são claros, tempo (clima) e periodo do dia. Então acho que podemos começar com essa análise

In [ ]:
df.describe()

In [ ]:
fixed_season_palette = {
    'Summer': '#440154',  # Roxo escuro (mesmo tom do primeiro gráfico)
    'Autumn': '#31688e',  # Azul/Verde escuro
    'Spring': '#21918c',  # Verde azulado
    'Winter': '#7ad151'   # Verde claro
}


In [ ]:

# Agrupando os dados de aluguel de bicicletas por estação (Seasons)
bikes_by_season = df.groupby('Seasons')['Rented_Bike_Count'].sum().reset_index()

# Ordenando para melhor visualização
bikes_by_season = bikes_by_season.sort_values(by='Rented_Bike_Count', ascending=False)

# Criando o gráfico de barras
plt.figure(figsize=(10, 6))
sns.barplot(x='Seasons',
            y='Rented_Bike_Count',
            data=bikes_by_season,
            palette='viridis',
            hue='Seasons',
            hue_order=['Summer', 'Autumn', 'Spring', 'Winter'],
            legend=True)

# Adicionando títulos e rótulos
plt.title('Total de Bicicletas Alugadas por Estação do Ano', fontsize=14)
plt.xlabel('Estação (Seasons)', fontsize=12)
plt.ylabel('Total de Bicicletas Alugadas', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Exibindo o gráfico
plt.tight_layout()
plt.show()

In [ ]:
# Agrupando os dados por data_parsed (datetime) e estação
bikes_daily_season = df.groupby(['Date_Parsed', 'Seasons'])['Rented_Bike_Count'].sum().reset_index()

# Como agrupamos por 'Date_Parsed', a ordenação cronológica funciona perfeitamente
bikes_daily_season = bikes_daily_season.sort_values(by='Date_Parsed')

# Criando o gráfico de barras diário colorido por estação
plt.figure(figsize=(15, 6))
sns.barplot(
    x='Date_Parsed',
    y='Rented_Bike_Count',
    hue='Seasons',
    hue_order=['Summer', 'Autumn', 'Spring', 'Winter'],  # Mantém a mesma ordem de legenda do primeiro gráfico
    data=bikes_daily_season,
    dodge=False,  # Mantém as barras na mesma posição temporal
    palette=fixed_season_palette
)

# Ajustando os eixos e títulos
plt.title('Total de Bicicletas Alugadas por Dia Colorido pela Estação do Ano', fontsize=14, fontweight='bold')
plt.xlabel('Data (Ordenada no Tempo)', fontsize=12)
plt.ylabel('Total de Bicicletas Alugadas', fontsize=12)

plt.xticks([])  # Oculta as labels individuais de cada dia para evitar sobreposição

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

parece que as pessoas não alugam tanto assim no inverno, mas também andar de bicicleta no frio ninguém merece. Vamos olhar com uma lupa um pouco maior

Os dados são coletados no começo do invero e assim vão passando. Apenas com essa visualização fica bem claro a crescente pós inverno. Mas tem um ponto curioso, mesmo nesses periodos tem alguns pontos de queda bem claro e eu gostaria de saber quais são essas datas e o que elas tem em comum. Vamos analisar finais de semana, feriados e datas comemorativas.

Busquei procurar os feriados comuns em seul:
| Data | O que é |
|---|---|
| 25/12/2017 | Natal (성탄절) |
| 01/01/2018 | Ano-Novo (신정) |
| 15/02/2018 | Véspera de Seollal |
| 16/02/2018 | Seollal / Ano-Novo Lunar (설날) |
| 17/02/2018 | Dia seguinte ao Seollal |
| 01/03/2018 | Dia do Movimento de Independência (삼일절) |
| 05/05/2018 | Dia das Crianças (어린이날) |
| 07/05/2018 | Feriado substituto do Dia das Crianças |
| 22/05/2018 | Aniversário de Buda (부처님 오신 날) |
| 06/06/2018 | Dia Memorial (현충일) |
| 15/08/2018 | Dia da Libertação (광복절) |
| 23/09/2018 | Véspera de Chuseok |
| 24/09/2018 | Chuseok / Ação de Graças coreana (추석) |
| 25/09/2018 | Dia seguinte ao Chuseok |
| 26/09/2018 | Feriado substituto do Chuseok |
| 03/10/2018 | Dia da Fundação Nacional (개천절) |
| 09/10/2018 | Dia do Hangeul (한글날) |

ref: https://time.is/calendar/2018/South_Korea

In [ ]:
# Criando um DataFrame auxiliar agrupado para plotar em milhares (10^3)
bikes_holiday_summary = df.groupby('Holiday')['Rented_Bike_Count'].sum().reset_index()

# 1. Gráfico de barras comparando dias com feriado vs dias normais (em milhares de aluguéis)
plt.figure(figsize=(8, 5))
sns.barplot(
    x='Holiday',
    y='Rented_Bike_Count',
    data=bikes_holiday_summary,
    palette='viridis',
    hue='Holiday',
    legend=False
)
plt.title('Total de Bicicletas Alugadas: Feriado vs. Dia Comum (em Milhares)', fontsize=12)
plt.xlabel('Tipo de Dia', fontsize=10)
plt.ylabel('Total de Aluguéis (em Milhares - 10³)', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Análise específica das datas dos feriados listados pelo usuário
# Dicionário mapeando as datas (DD/MM/AAAA) para o nome amigável do feriado
feriados_mapeados = {
    '25/12/2017': 'Natal',
    '01/01/2018': 'Ano-Novo',
    '15/02/2018': 'Véspera de Seollal',
    '16/02/2018': 'Seollal / Ano-Novo Lunar',
    '17/02/2018': 'Dia seguinte ao Seollal',
    '01/03/2018': 'Dia do Movimento de Independência',
    '05/05/2018': 'Dia das Crianças',
    '07/05/2018': 'Feriado substituto do Dia das Crianças',
    '22/05/2018': 'Aniversário de Buda',
    '06/06/2018': 'Dia Memorial',
    '15/08/2018': 'Dia da Libertação',
    '23/09/2018': 'Véspera de Chuseok',
    '24/09/2018': 'Chuseok / Ação de Graças',
    '25/09/2018': 'Dia seguinte ao Chuseok',
    '26/09/2018': 'Feriado substituto do Chuseok',
    '03/10/2018': 'Dia da Fundação Nacional',
    '09/10/2018': 'Dia do Hangeul'
}

# Agrupando os aluguéis totais por data original (formato DD/MM/AAAA)
daily_rentals = df.groupby('Date')['Rented_Bike_Count'].sum().reset_index()

# Filtrando apenas as datas que correspondem aos feriados e adicionando os nomes
daily_rentals_holidays = daily_rentals[daily_rentals['Date'].isin(feriados_mapeados.keys())].copy()
daily_rentals_holidays['Nome_Feriado'] = daily_rentals_holidays['Date'].map(feriados_mapeados)

# Ordenando pelo volume de aluguéis para facilitar a análise visual
daily_rentals_holidays = daily_rentals_holidays.sort_values(by='Rented_Bike_Count', ascending=False)

# Plotando o gráfico desses feriados específicos
plt.figure(figsize=(12, 6))
sns.barplot(
    x='Rented_Bike_Count',
    y='Nome_Feriado',
    data=daily_rentals_holidays,
    palette='viridis',
    hue='Nome_Feriado',
    legend=False
)
plt.title('Total de Bicicletas Alugadas em Cada Feriado de Seul', fontsize=14)
plt.xlabel('Total de Aluguéis no Dia', fontsize=12)
plt.ylabel('Feriado', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

Algo bem curiso, o gráfico contando o total de alugueis em feriados é bem baixo em relação aos dias de não feriado. Isso já era bem esperado visto que temos muitos menos feriados no ano. Mas o gráfico mais especifico pegando as datas de feriado feito na coleta externa mostra que tem dias de feriados em que os alugueis são bem comuns. Minha pergunta inicial para os feriados era saber se o alugal era mais baixo, porém vi o contrário. Na maioria dos dias é mt requisitado. Ainda fica a questão, saber se o dia da semana em espefico impacta no numero de alugueis.

In [ ]:
# Criando a coluna do dia da semana (0 = Segunda, 6 = Domingo)
df['Day_of_Week'] = df['Date_Parsed'].dt.dayofweek

# Criando a variável categórica para diferenciar Dia de Semana de Final de Semana
# 5 representa Sábado e 6 representa Domingo
df['Day_Type'] = df['Day_of_Week'].apply(lambda x: 'Weekend' if x >= 5 else 'Weekday')

# Vamos ver a distribuição dos dados e uma pequena amostra
print("Distribuição dos registros por Tipo de Dia:")
display(df['Day_Type'].value_counts())

print("\nAmostra das colunas criadas:")
display(df[['Date_Parsed', 'Day_of_Week', 'Day_Type']].drop_duplicates().sort_values(by='Date_Parsed').head(10))

In [ ]:
# --- GRÁFICO COMPARATIVO GERAL (TOTAL ACUMULADO) ---
# Agrupando o total geral por tipo de dia
total_by_day_type = df.groupby('Day_Type')['Rented_Bike_Count'].sum().reset_index()
# Passando os valores para milhares para facilitar a leitura
total_by_day_type['Total_K'] = total_by_day_type['Rented_Bike_Count'] / 1000

plt.figure(figsize=(8, 5))
sns.barplot(
    x='Day_Type',
    y='Total_K',
    data=total_by_day_type,
    palette='viridis',
    hue='Day_Type',
    legend=False
)
plt.title('Comparativo Geral: Total de Aluguéis em Dias de Semana vs. Finais de Semana', fontsize=12, fontweight='bold')
plt.xlabel('Tipo de Dia', fontsize=10)
plt.ylabel('Total de Aluguéis (em Milhares - 10³)', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Agrupando os dados diários contendo o total de aluguéis, tipo de dia e a estação do ano
daily_summary = df.groupby(['Date_Parsed', 'Day_Type', 'Seasons'])['Rented_Bike_Count'].sum().reset_index()

# Separando os dados de dias de semana e finais de semana
weekdays_data = daily_summary[daily_summary['Day_Type'] == 'Weekday'].sort_values(by='Date_Parsed')
weekends_data = daily_summary[daily_summary['Day_Type'] == 'Weekend'].sort_values(by='Date_Parsed')

# --- GRÁFICOS TEMPORAIS POR DIA (COLORIDOS POR ESTAÇÃO) ---
# Gráfico para Dias de Semana
plt.figure(figsize=(15, 5))
sns.barplot(
    x='Date_Parsed',
    y='Rented_Bike_Count',
    hue='Seasons',
    hue_order=['Summer', 'Autumn', 'Spring', 'Winter'],
    data=weekdays_data,
    dodge=False,
    palette=fixed_season_palette
)
plt.title('Aluguel Diário de Bicicletas - Dias de Semana (Colorido por Estação)', fontsize=14, fontweight='bold')
plt.xlabel('Data', fontsize=12)
plt.ylabel('Total de Aluguéis', fontsize=12)
plt.xticks([])
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Gráfico para Finais de Semana
plt.figure(figsize=(15, 5))
sns.barplot(
    x='Date_Parsed',
    y='Rented_Bike_Count',
    hue='Seasons',
    hue_order=['Summer', 'Autumn', 'Spring', 'Winter'],
    data=weekends_data,
    dodge=False,
    palette=fixed_season_palette
)
plt.title('Aluguel Diário de Bicicletas - Finais de Semana (Colorido por Estação)', fontsize=14, fontweight='bold')
plt.xlabel('Data', fontsize=12)
plt.ylabel('Total de Aluguéis', fontsize=12)
plt.xticks([])
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Agrupando os dados diários contendo o total de aluguéis e o tipo de dia
bikes_daily_daytype = df.groupby(['Date_Parsed', 'Day_Type'])['Rented_Bike_Count'].sum().reset_index()

# Garantindo a ordenação cronológica
bikes_daily_daytype = bikes_daily_daytype.sort_values(by='Date_Parsed')

# Criando o gráfico de barras diário colorido por tipo de dia (dia de semana vs final de semana)
plt.figure(figsize=(15, 6))
sns.barplot(
    x='Date_Parsed',
    y='Rented_Bike_Count',
    hue='Day_Type',
    hue_order=['Weekday', 'Weekend'],
    data=bikes_daily_daytype,
    dodge=False,  # Mantém as barras na mesma ordem temporal
    palette='Set2'
)

# Ajustando títulos e eixos
plt.title('Total de Bicicletas Alugadas por Dia (Colorido por Dia de Semana vs. Final de Semana)', fontsize=14, fontweight='bold')
plt.xlabel('Data (Ordenada no Tempo)', fontsize=12)
plt.ylabel('Total de Bicicletas Alugadas', fontsize=12)

# Ocultando os rótulos do eixo X para evitar sobreposição
plt.xticks([])

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

Bem ficou claro que não consigo relacionar tão bem os dias da semana com o total de vendas de bicicleta, pelo menos não de uma forma separada. Possivelmente tomando apenas a data em que o total de vendas seja menor que 5000 não vai me dar uma ideia tão clara. Possivelmente não é o dia que está impactando no volume das vendas, mas sim as caracteristicas do que ocorre naquele dia. Isso claro não deve valer tanto para feriados, em que podemos associar o volume de alugueis aos dias em si. Como sabemos que no inverno já temos um volume de alugueis mais baixos talvez caracteristicas de clima semelhantes nos outros dias do ano possa impactar nos alugueis nesses dias. Saberei ao analisar as variáveis numericas. Para finalizar, com o mesmo gráfico de barras posso verificar a questão do sistema

In [ ]:
# Agrupando os dados diários contendo o total de aluguéis e o funcionamento do sistema
bikes_daily_functioning = df.groupby(['Date_Parsed', 'Functioning_Day'])['Rented_Bike_Count'].sum().reset_index()

# Garantindo a ordenação cronológica
bikes_daily_functioning = bikes_daily_functioning.sort_values(by='Date_Parsed')

# Criando o gráfico de barras diário colorido por status de funcionamento
plt.figure(figsize=(15, 6))
sns.barplot(
    x='Date_Parsed',
    y='Rented_Bike_Count',
    hue='Functioning_Day',
    hue_order=['Yes', 'No'],
    data=bikes_daily_functioning,
    dodge=False,
    palette='Set2'
)

# Ajustando títulos e eixos
plt.title('Total de Bicicletas Alugadas por Dia (Colorido por Funcionamento do Sistema: Ativo vs Inativo)', fontsize=14, fontweight='bold')
plt.xlabel('Data (Ordenada no Tempo)', fontsize=12)
plt.ylabel('Total de Bicicletas Alugadas', fontsize=12)

# Ocultando os rótulos do eixo X para evitar sobreposição
plt.xticks([])

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Agrupando os dados diários contendo o total de aluguéis e o funcionamento do sistema
bikes_daily_functioning = df.groupby(['Date_Parsed', 'Functioning_Day'])['Rented_Bike_Count'].sum().reset_index()
bikes_daily_functioning[bikes_daily_functioning["Functioning_Day"] == "No"]

Como eu imaginava, em dia que não tem funcionamento nem contagem não tem. Agora vamos analisar as variveis numericas, elas devem responder minha questão inicial. Porque tem dias no meio do ano tão baixos qunato no inverno